# Tutorial 2, Q6a–c: diet problem

**The problem.** $x_i \ge 0$ is the number of units of food $i$ ($i = 1, \dots, 10$) in the diet. Table 2 of
the sheet gives each food's vitamin A–D and fat content, cost and attractiveness; Table 1 gives the limits:
at least 100, 120, 200 and 80 units of vitamins A, B, C and D, and at most 300 units of fat.

- (a) minimise the cost;
- (b) maximise the attractiveness (the problem is unbounded);
- (c) both objectives again with maximum = minimum + 20% limits on vitamins and fat (each vitamin between
  its minimum and 1.2 × minimum; fat between 300/1.2 = 250 and 300).

For (b) and (c) the attractiveness is maximised, so the printed `fval` is **minus** the attractiveness. After
(a) and (c) the notebook lists the binding constraints (the rows of `A_ub` with no slack, numbered as in the
answer sheet).

**How to run it.** *Run → Run All Cells*, or press Shift + Enter in each cell while you read. Nothing to install: SciPy is built into the browser's Python (the first import takes a few seconds).

**Expected results.** (a) cost `fval = 93.25` with $x_2 = 9.09$, $x_5 = 1.43$, $x_{10} = 2.27$, constraints
1–3 binding; (b) *The problem is unbounded*; (c) attractiveness 149.66 (`fval = -149.66`), constraints 2, 4,
5, 8 and 10 binding; and with the cost objective under the same limits, cost `fval = 136.52`.

Script name says 5: the question was Q5 on an earlier sheet.

*Original script: `5 Source Files/Python/Question_5.py`. Runs in the browser.*

Load `linprog`, SciPy's linear-programming solver. It always **minimises** $c^T x$ subject to `A_ub` $x \le$ `b_ub`, and by default every variable is $\ge 0$, so the non-negativity constraints need no rows of their own.

In [ ]:
from scipy.optimize import linprog

**Part A: minimise the cost** $\sum_i c_i x_i$ (the *Cost* column of Table 2). It is already a minimisation, so the costs go in unchanged.

In [ ]:
# Part A

# costs (row vector): we minimise cost, so linprog takes them as they are
c = [10, 6, 8, 16, 8, 3, 6, 8, 12, 12]

`linprog` takes only $\le$ rows, so each *at least* vitamin constraint is multiplied by $-1$: $\sum_i (Va)_i x_i \ge 100$ becomes $-\sum_i (Va)_i x_i \le -100$. Fat is already a $\le$ constraint. A '-' in Table 2 is a 0.

In [ ]:
# linear inequalities 
# (no. rows in A = no. rows in b)
A_ub = [[-18, -8, -22, -6, 0, 0, -8, -4, 0, -12],    # vitamin A >= 100 (as -A <= -100)
        [-4, -6, -10, -12, -14, 0, 0, 0, -6, -20],   # vitamin B >= 120
        [0, -22, 0, 0, 0, -20, 0, -8, -10, 0],       # vitamin C >= 200
        [-6, -14, 0, -6, -10, -8, -6, 0, -12, -4],   # vitamin D >= 80
        [0, 0, 8, 0, 0, 6, 20, 40, 0, 0]]            # fat <= 300
b_ub = [-100, -120, -200, -80, 300]   

Solve Part A and print the diet.

In [ ]:
#  call the linprog function
results = linprog(c, A_ub, b_ub)

print(results.message)
# if results are a success then print out the optimal solution
if results.success:
    for k in range(len(results.x)):
        print(f"x[{k+1}] = {results.x[k]:.2f}")
    print(f"fval = {results.fun:.2f}")  
        

A constraint is **binding** when it holds with equality, i.e. its slack (`b_ub` minus `A_ub` $x$) is 0. `results.slack` has one entry per row of `A_ub`; the tolerance `1e-6` allows for rounding.

In [ ]:
# binding constraints: rows of A_ub with no slack (numbered from 1, as in the answer sheet)
for k, s in enumerate(results.slack):
    if abs(s) < 1e-6:
        print(f"Constraint {k+1} is binding")

**Part B: maximise the attractiveness** (the last column of Table 2), so the values are negated. The constraints are those of Part A.

In [ ]:
# Part B
# attractiveness (row vector): to maximise it, linprog minimises the negatives
a = [-6, -1, -5, -5, -6, -4, -10, -10, -6, -7]

Expect *The problem is unbounded*: the foods with no fat (1, 2, 4, 5, 9 and 10) only help the vitamin minimums, so they can be increased without limit and the attractiveness grows for ever.

In [ ]:
#  call the linprog function
results = linprog(a, A_ub, b_ub)

print(results.message)
# if results are a success then print out the optimal solution
if results.success:
    for k in range(len(results.x)):
        print(f"x[{k+1}] = {results.x[k]:.2f}")
    print(f"fval = {results.fun:.2f}")       

**Part C: add the maximums.** Each nutrient now gets two rows: $-$content $\le -$minimum, then content $\le$ maximum. Rows 1–2 are vitamin A, 3–4 vitamin B, 5–6 vitamin C, 7–8 vitamin D and 9–10 fat.

In [ ]:
# Part C: each row pair is  -content <= -minimum  and  content <= maximum
# linear inequalities 
# (no. rows in A = no. rows in b)
A_ub = [[-18, -8, -22, -6, 0, 0, -8, -4, 0, -12],    # 1  vitamin A >= 100
        [18, 8, 22, 6, 0, 0, 8, 4, 0, 12],           # 2  vitamin A <= 120
        [-4, -6, -10, -12, -14, 0, 0, 0, -6, -20],   # 3  vitamin B >= 120
        [4, 6, 10, 12, 14, 0, 0, 0, 6, 20],          # 4  vitamin B <= 144
        [0, -22, 0, 0, 0, -20, 0, -8, -10, 0],       # 5  vitamin C >= 200
        [0, 22, 0, 0, 0, 20, 0, 8, 10, 0],           # 6  vitamin C <= 240
        [-6, -14, 0, -6, -10, -8, -6, 0, -12, -4],   # 7  vitamin D >= 80
        [6, 14, 0, 6, 10, 8, 6, 0, 12, 4],           # 8  vitamin D <= 96
        [0, 0, -8, 0, 0, -6, -20, -40, 0, 0],        # 9  fat >= 300/1.2 = 250
        [0, 0, 8, 0, 0, 6, 20, 40, 0, 0]             # 10 fat <= 300
        ]
b_ub = [-100, 100*1.2, -120, 120*1.2, -200, 200*1.2, -80, 80*1.2, -300/1.2, 300]

Maximise the attractiveness (`a` from Part B) under the new limits.

In [ ]:
#  call the linprog function
results = linprog(a, A_ub, b_ub)

print(results.message)
# if results are a success then print out the optimal solution
if results.success:
    for k in range(len(results.x)):
        print(f"x[{k+1}] = {results.x[k]:.2f}")
    print(f"fval = {results.fun:.2f}")  
        

The binding rows for Part C, numbered as in the comments of the `A_ub` cell above.

In [ ]:
# binding constraints: rows of A_ub with no slack (numbered from 1, as in the answer sheet)
for k, s in enumerate(results.slack):
    if abs(s) < 1e-6:
        print(f"Constraint {k+1} is binding")

### Part C with the cost objective

The minimum-cost diet from (a) breaks the new limits (it has no fat and 150.6 units of vitamin D, above 1.2 × 80 = 96), so the cost rises from £93.25 to £136.52.

In [ ]:
#  call the linprog function: minimise cost
results = linprog(c, A_ub, b_ub)

print(results.message)
# if results are a success then print out the optimal solution
if results.success:
    for k in range(len(results.x)):
        print(f"x[{k+1}] = {results.x[k]:.2f}")
    print(f"fval = {results.fun:.2f}")  
        

The binding rows for the minimum-cost diet under the Part C limits.

In [ ]:
# binding constraints: rows of A_ub with no slack (numbered from 1, as in the answer sheet)
for k, s in enumerate(results.slack):
    if abs(s) < 1e-6:
        print(f"Constraint {k+1} is binding")

## What do you notice?

1. Read the binding rows of Part C (2, 4, 5, 8, 10) with the comments in the `A_ub` cell. Which limits
   stop the attractiveness from growing?
2. Why is Part C bounded when Part B was not?
3. Why can the cost under the Part C limits (£136.52) not be lower than in Part A (£93.25)?

<details><summary>Answers</summary>

1. Vitamins A, B and D at their **maximum**, vitamin C at its **minimum** and fat at its **maximum** (300).
   The cost version binds at rows 1, 3, 5, 8 and 9: vitamins A, B and C at their minimum, vitamin D at its
   maximum and fat at its minimum (250).
2. Every food contains at least one vitamin, and every vitamin now has a maximum, so no $x_i$ can grow
   without limit.
3. Part C only **adds** constraints, so its feasible region lies inside Part A's and the minimum cost cannot
   fall. It rises because Part A's diet is no longer allowed: it has no fat and 150.6 units of vitamin D.
</details>